# HCM AI Challenge: Colab setup

This notebook keeps mutable state on Google Drive. Run it once per fresh Colab runtime before indexing or search. It never reads or writes a local `.env` file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# Change only these two paths for your own Drive/repository checkout.
DRIVE_ROOT = Path('/content/drive/MyDrive/HCM-AI-Challenge-2026')
REPO_ROOT = Path('/content/HCM-AI-Challenge-2026')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError(
        f'Clone or upload this repository to {REPO_ROOT}, then rerun this cell. '
        'Do not put the editable checkout inside a generated artifact folder.'
    )

In [ ]:
%cd {REPO_ROOT}
# Install runtime extras only in the ephemeral Colab VM. Model/download caches remain on Drive below.
!pip -q install -e ".[dev,retrieval,models,ocr,asr,gemini]"

In [ ]:
DATA_ROOT = DRIVE_ROOT / 'data'
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
MODEL_CACHE = DRIVE_ROOT / 'model_cache'
for path in (DATA_ROOT, ARTIFACT_ROOT, OUTPUT_ROOT, MODEL_CACHE):
    path.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'DATA_ROOT': str(DATA_ROOT),
    'ARTIFACT_ROOT': str(ARTIFACT_ROOT),
    'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'MODEL_CACHE': str(MODEL_CACHE),
    'HF_HOME': str(MODEL_CACHE / 'huggingface'),
    'TRANSFORMERS_CACHE': str(MODEL_CACHE / 'huggingface' / 'hub'),
})
print({key: os.environ[key] for key in ('DATA_ROOT', 'ARTIFACT_ROOT', 'OUTPUT_ROOT', 'MODEL_CACHE')})

In [ ]:
# Optional: set GOOGLE_API_KEY from Colab Secrets. Never print it or save it to Drive.
try:
    from google.colab import userdata
    secret = userdata.get('GOOGLE_API_KEY')
    if secret:
        os.environ['GOOGLE_API_KEY'] = secret
except Exception:
    # Local/offline runs retain the deterministic heuristic fallback.
    pass

print('Gemini key available:', bool(os.environ.get('GOOGLE_API_KEY')))

In [ ]:
from hcm_ai.runtime import detect_capabilities, resolve_profile

PROFILE_REQUEST = 'auto'  # cpu, balanced_gpu, paper_gpu, or auto
PROFILE = resolve_profile(PROFILE_REQUEST)
os.environ['HCM_AI_PROFILE'] = PROFILE
capabilities = detect_capabilities()
print('Resolved profile:', PROFILE)
print(capabilities)

# The profile can safely downgrade on a CPU runtime. Heavy models are loaded lazily by commands that use them.